# `Graph2Vec`

# Packages

In [31]:
# pip install networkx numpy==1.22.4 pandas matplotlib seaborn karateclub

In [32]:
import inspect
from karateclub import Graph2Vec
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import networkx as nx
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns
from typing import Union, get_args

# Directories

In [33]:
DATAFRAME = "D:/DDesktop/_code/canada/lincs_analysis/dataframes/"
GRAPH = "D:/DDesktop/_code/canada/lincs_analysis/graphs/"
GRAPH_BASE = "D:/DDesktop/_code/canada/lincs_analysis/graphs/base/"
GRAPH_PERTURBAGEN = "D:/DDesktop/_code/canada/lincs_analysis/graphs/perturbagen/"
SIGNATURE = "D:/DDesktop/_code/canada/lincs_analysis/lincs_signatures/"
STRING = "D:/DDesktop/_code/canada/lincs_analysis/string/"
PLOT = "D:/DDesktop/_code/canada/lincs_analysis/plots/"

# Functions

In [34]:
def func_param_typecheck(function, *args, **kwargs) -> None:
    '''
    Checks the types of parameters for a given function based on parameter annotations.

    ### Parameters

    `function`: Any

    - The function whose parameters are being checked.

    `*args`: tuple

    - Tuple of positional parameters for given function, can be any type.

    `**kwargs` : dict

    - Dictionary of keyword arguments passed to the given function, where keys are strings and values are any type.

    ### Raises

    `TypeError`

    - Error raised if any parameter does not match its annotated type
    '''

    ## FUNCTION

    # Get function signature
    function_signature = inspect.signature(function)
    # Get function arguments
    function_arguments = function_signature.bind(*args, **kwargs)

    # Iterate through function parameters and input data types
    for parameter_name, parameter_type in function_arguments.arguments.items():
        # Get expected parameter type from annotation
        expected_type = function_signature.parameters[parameter_name].annotation
        # Check the expected type isn't empty
        if expected_type != inspect.Parameter.empty:
            # Check if parameter annotations are in a Union
            if hasattr(expected_type, '__origin__') and expected_type.__origin__ is Union:
                # Type check
                if not isinstance(parameter_type, get_args(expected_type)):
                    # Extract list of allowed types
                    allowed_types = ', '.join(etype.__name__ for etype in get_args(expected_type))
                    # Raise error
                    raise TypeError(f'Parameter \'{parameter_name}\' must be one of types: \'{allowed_types}\', ' 
                                    f'but got \'{type(parameter_type).__name__}\'')
            # Else type check single annotated type
            elif not isinstance(parameter_type, expected_type):
                # Raise error
                raise TypeError(f'Parameter \'{parameter_name}\' must be of type \'{expected_type.__name__}\', '
                                f'but got \'{type(parameter_type).__name__}\'')

def graph_save(graph: nx.Graph, graph_name: str, path: str = os.getcwd(), graphml: bool = False, report: bool = False) -> None:
    '''
    Saves a NetworkX graph object to specified location using the pickle package OR as a .gml file for downstream analysis.

    ### Parameters

    `graph`: nx.Graph

    > NetworkX graph object to be saved.

    `graph_name`: str

    > Name for saved file.

    `path`: str (default = current working directory)

    > Save location.

    ### Returns

    Pickled graph object
    '''

    ## CHECK

    # Type
    func_param_typecheck(graph_save, graph, graph_name, path, graphml, report)

    ## FUNCTION

    # Check graphml parameter
    if graphml == False:
        # Pickle graph
        with open(f'{path}/{graph_name}.pkl', 'wb') as f:
            pickle.dump(graph, f)
    else:
        # Save graph as .gml file
        nx.write_graphml(graph, path = f'{path}/{graph_name}.gml')

    ## REPORT

    if report == True:
        # Statistics
        # Print
        print('>> graph_save')
        if graphml == False:
            print(f'NetworkX graph object saved to {path}/{graph_name}.pkl')
        else:
            print(f'NetworkX graph object saved to {path}/{graph_name}.gml')
        print()

def graph_load(path: str, graphml: bool = False, report: bool = False) -> nx.Graph:
    '''
    Loads a pickled NetworkX graph object.

    ### Parameters

    `path`: str

    > Location of the pickled graph file.

    ### Returns

    `graph`: nx.Graph

    > NetworkX graph object.
    '''

    ## CHECK

    # Type
    func_param_typecheck(graph_load, path, graphml, report)

    ## FUNCTION

    # Check graphml parameter
    if graphml == False:
        with open(f'{path}', 'rb') as f:
            graph = pickle.load(f)
    else:
        graph = nx.read_graphml(path)
        graph = nx.relabel_nodes(graph, lambda x: int(x))

    ## REPORT

    if report == True:
        # Statistics
        num_nodes = graph.number_of_nodes()
        num_edges = graph.number_of_edges()
        # Print
        print('>> graph_load')
        if graphml == False:
            print(f'Pickled NetworkX graph object with {num_nodes:,} nodes and {num_edges:,} edges loaded from {path}')
        else:
            print(f'GraphML NetworkX graph object with {num_nodes:,} nodes and {num_edges:,} edges loaded from {path}')
        print()

    return graph

def graph_visualise_network(graph: nx.Graph, pos: dict, cell_line: str, drug: str, dosage: str, timepoint: int, figx: Union[float, int] = 10, figy: Union[float, int] = 10, cmap: str = 'seismic', vmin: Union[float, int] = 0.1, vmax: Union[float, int] = 0.1, edge_factor: Union[float, int] = 500, edge_colour: str = 'black', edge_alpha: float = 0.5, colourbar : bool = True) -> Figure:
    '''
    Plots a NetworkX graph object as a standard network. Must be passed a graph with nested cell line, drug, dosage and timepoint node attributes. Allows for some nodes to lack node attributes.

    ### Parameters

    `graph`: nx.Graph

    > NetworkX graph object to be visualised.

    `pos`: dict

    > Positional data for graph object.

    `cell_line`: str

    > Cell line used to filter graph node attributes e.g. 'HT29'

    `drug`: str

    > Drug name used to filter graph node attributes e.g. 'ibuprofen'

    `dosage`: str

    > Dosage value used to filter graph node attributes, in uM e.g. '10'

    `timepoint`: int

    > Timepoint value used to filter graph node attributes e.g. 6

    `figx`: float, int (default = 10)

    > X-axis size of plot.

    `figy`: float, int (default = 10)

    > Y-axis size of plot.

    `cmap`: str (default = 'seismic')

    > Colourmap used to represent CD-coefficient node attribute value in plot. Diverging colourmap is recommended.    

    `vmin`: float, int (default = -0.1)

    > Minimum value for `colourmap`

    `vmax`: float, int (default = 0.1)

    > Maximum value for `colourmap`

    `edge_factor`: float, int (default = 500)

    > Factor by which the edge weights are divided for visualisation. Higher value is recommended if using combined score directly from STRING data.

    `edge_colour`: str (default = 'black')

    > Colour for plotted edges.

    `edge_alpha`: float (default = 0.5)

    > Alpha value for plotted edges.

    `colourbar`: bool (default = True)

    > Adds a colourbar to the plot.

    ### Returns

    A NetworkX visualisation of a graph, overlayed with LINCS1000 CD-coefficient values from a defined filter of attributes.

    '''

    ## CHECK

    # Type
    func_param_typecheck(graph_visualise_network, graph, pos, cell_line, drug, dosage, timepoint, figx, figy, cmap, vmin, vmax, edge_factor, edge_colour, edge_alpha, colourbar)

    ## FUNCTION
    
    # Initialise lists
    nodes_with_vals = []
    nodes_without_vals = []
    values = [] 
    edge_weights = [graph[u][v]['weight']/edge_factor for u, v in graph.edges()]

    # Iterate through nodes
    for node in graph.nodes:
        # Filter for all attributes
        filter_value = (graph.nodes[node]
                        .get(cell_line, {})
                        .get(drug, {})
                        .get(dosage, {})
                        .get(timepoint))

        # Check filter_value
        if filter_value is not None:
            # Append node to list
            nodes_with_vals.append(node)
            # Append value to list
            values.append(filter_value)
        else:
            # Append node to list
            nodes_without_vals.append(node)

    ## PLOT

    # Initialise plot
    plt.figure(figsize = (figx, figy))
    
    # Draw edges
    plot_edges = nx.draw_networkx_edges(graph, pos = pos, width = edge_weights, edge_color = edge_colour, alpha = edge_alpha)
    # Draw nodes without values
    plot_nodes_without = nx.draw_networkx_nodes(graph, pos = pos, nodelist = nodes_without_vals, node_color = 'lightgrey', edgecolors = 'black')
    # Draw nodes with values
    plot_nodes_with = nx.draw_networkx_nodes(graph, pos = pos, nodelist = nodes_with_vals, node_color = values, cmap = 'seismic_r', vmin = vmin, vmax = vmax, edgecolors = 'black')

    # Check colorbar parameter
    if colourbar == True:
        # Generate colorbar
        cbar = plt.colorbar(plot_nodes_with, label = 'CD-coefficient', orientation = 'horizontal', fraction= 0.02)
        # Set position
        cbar.ax.set_position([0.75, 0.85, 0.15, 0.02])

    # Remove border
    plt.box(False)

    # Define title
    plt.title(f'Human STRING PPI network w/ CD-coefficient values from {drug}-treated {cell_line} cells ({dosage}uM) at {timepoint}h')
    # Show plot
    plt.show()

# Analysis

## Load Graphs

In [35]:
# Initialise lists
list_graphs = []
list_graph_names = []

# Iterate through GRAPH_PERTURBAGEN directory
for file in os.listdir(GRAPH_PERTURBAGEN):
    # Check for `.pkl` file
    if file.endswith('.pkl'):
        # Define file_path
        file_path = os.path.join(GRAPH_PERTURBAGEN, file)
        # Load graph
        G = graph_load(file_path, graphml = False, report = False)
        # Get graph name
        graph_name = file.split('.')[0]
        # Append graph to `list_graphs`
        list_graphs.append(G)
        # Append graph name
        list_graph_names.append(graph_name)

print('list_graph generated')

list_graph generated


## Get Embeddings

In [36]:
# Initialise Graph2Vec
model = Graph2Vec()

# Train Graph2Vec
model.fit(list_graphs)

In [37]:
# Extract embeddings
graph_embeddings = model.get_embedding()
# Convert to dataframe
df_embeddings = pd.DataFrame(graph_embeddings, index = list_graph_names)
# Save data
df_embeddings.to_csv(DATAFRAME + 'df_embeddings.csv')
# Show data
df_embeddings.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
HT29_ceramide_10uM_24,0.850119,1.459740,0.539817,-0.507828,-0.268028,-0.750147,-0.407783,0.447334,-1.157096,-0.564132,...,0.230500,-0.332161,0.298470,0.739336,0.142748,1.014386,-0.438541,0.300555,0.392092,0.756250
HT29_ceramide_10uM_6,0.873862,1.481295,0.540093,-0.519665,-0.263161,-0.761324,-0.415184,0.440858,-1.195708,-0.584396,...,0.238659,-0.333636,0.299644,0.753003,0.142622,1.026267,-0.444824,0.290392,0.403193,0.766710
HT29_chlorphensin_10uM_24,0.877841,1.495315,0.546250,-0.522062,-0.253496,-0.765938,-0.423253,0.442254,-1.216106,-0.597567,...,0.240839,-0.331476,0.302620,0.765402,0.152317,1.040692,-0.450484,0.282319,0.415036,0.775801
HT29_chlorphensin_10uM_6,0.883302,1.495000,0.541848,-0.522730,-0.249061,-0.765856,-0.423721,0.436127,-1.225602,-0.603325,...,0.242739,-0.329017,0.296112,0.764697,0.153701,1.042446,-0.448799,0.279723,0.413629,0.773975
HT29_dihydroobliquin_10uM_24,0.848480,1.446980,0.531792,-0.507678,-0.246504,-0.743218,-0.410661,0.429256,-1.172351,-0.573141,...,0.230495,-0.321947,0.291424,0.744427,0.142826,1.009413,-0.434494,0.278356,0.399672,0.749533


In [38]:
print('EMBEDDINGS GENERATED')

EMBEDDINGS GENERATED


In [40]:
!pip list

Package            Version
------------------ -----------
asttokens          3.0.0
colorama           0.4.6
comm               0.2.2
contourpy          1.2.1
cycler             0.12.1
debugpy            1.8.13
decorator          4.4.2
exceptiongroup     1.2.2
executing          2.2.0
fonttools          4.56.0
gensim             4.3.3
ipykernel          6.29.5
ipython            8.34.0
jedi               0.19.2
joblib             1.4.2
jupyter_client     8.6.3
jupyter_core       5.7.2
karateclub         1.3.3
kiwisolver         1.4.8
Levenshtein        0.27.1
matplotlib         3.8.4
matplotlib-inline  0.1.7
nest-asyncio       1.6.0
networkx           2.6.3
numpy              1.22.4
packaging          24.2
pandas             1.3.5
parso              0.8.4
pillow             11.1.0
pip                21.2.3
platformdirs       4.3.6
prompt_toolkit     3.0.50
psutil             7.0.0
pure_eval          0.2.3
Pygments           2.19.1
PyGSP              0.5.1
pyparsing          3.2.1
python

You should consider upgrading via the 'D:\DDesktop\_code\.venv_graph2vec_310\Scripts\python.exe -m pip install --upgrade pip' command.
